In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

# --- 1. Load Data ---
# Ensure train.csv and test.csv are uploaded to the Files section on the left
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Save Id for submission and drop from features
test_ids = test['Id']
train.drop('Id', axis=1, inplace=True)
test.drop('Id', axis=1, inplace=True)

# --- 2. Preprocessing ---
# Log Transform Target
y_train = np.log1p(train['SalePrice'])
train_features = train.drop('SalePrice', axis=1)

# Concatenate for consistent processing
all_data = pd.concat([train_features, test]).reset_index(drop=True)

# Impute Categorical Features (where NA means 'None')
cols_na_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                'MasVnrType']
for col in cols_na_none:
    all_data[col] = all_data[col].fillna('None')

# Impute Numerical Features (where NA means 0)
cols_na_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars',
                'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
                'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in cols_na_zero:
    all_data[col] = all_data[col].fillna(0)

# Impute LotFrontage with Neighborhood Median
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median()))

# Impute remaining with Mode
cols_mode = ['MSZoning', 'Utilities', 'Functional', 'Electrical',
             'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType']
for col in cols_mode:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# Feature Engineering: Convert numerical categories to string
all_data['MSSubClass'] = all_data['MSSubClass'].apply(str)
all_data['OverallCond'] = all_data['OverallCond'].astype(str)
all_data['YrSold'] = all_data['YrSold'].astype(str)
all_data['MoSold'] = all_data['MoSold'].astype(str)

# One-Hot Encoding
all_data = pd.get_dummies(all_data)

# Split back into Train and Test
X_train = all_data.iloc[:len(y_train), :]
X_test = all_data.iloc[len(y_train):, :]

# --- 3. Modeling ---
print("Training model...")
gbr = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.05,
                                max_depth=4, max_features='sqrt',
                                min_samples_leaf=15, min_samples_split=10,
                                loss='huber', random_state=42)
gbr.fit(X_train, y_train)

# --- 4. Prediction & Submission ---
print("Predicting...")
y_pred = gbr.predict(X_test)
y_pred_final = np.expm1(y_pred) # Reverse the log transform

# Create DataFrame
submission = pd.DataFrame({'Id': test_ids, 'SalePrice': y_pred_final})

# Save to CSV
submission.to_csv('submission.csv', index=False)
print("Success! 'submission.csv' has been created.")

Training model...
Predicting...
Success! 'submission.csv' has been created.
